# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/imatiq/ML_Internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/imatiq/ML_Internship"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found -- are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter
Starter data found. You're ready.


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

My ML-08/ML-09 archetype clustering (k=6, K-Means) is the base for this playbook. Each
archetype gets one standing action and one plain-language reason, ordered by the archetype's
own decline rate (highest first), and within an archetype by visibility -- the same
"review the biggest, riskiest group first" logic as my ML-07 rule.

| Archetype | Decline rate | n | Suggested action | Why |
|---|---|---|---|---|
| Stale Heavyweights | 0.65 | 2,887 | `refresh` | Long, old-updated, still-visible pages -- the content itself needs a rewrite/refresh pass first. |
| Aging Page-One (Engagement Gap) | 0.62 | 5,925 | `refresh_title_and_meta` | Ranking near page 1 but stale with thin engagement -- title/meta refresh is the cheapest high-leverage fix. |
| Young & Slipping | 0.61 | 11,058 | `monitor_closely` | Newer pages already showing risk despite low staleness -- watch, don't rewrite yet. |
| Established Performers | 0.41 | 7,352 | `monitor` | Below base rate, moderate age/position -- routine check-ins only. |
| Low-Demand Long-Tail | 0.37 | 2,638 | `low_priority` | Deep position, low impressions -- not worth review time unless demand shifts. |
| Near-Zero-Traffic | 0.14 | 140 | `deprioritize` | Almost no impressions -- lowest decline rate; quiet, not failing. |

Reason codes attached to every row (`action_reason` column) are one plain sentence, not a
code -- the goal here is a human-trusted queue, and short prose reads faster than a symbol
table for a one-line justification a reviewer skims once and moves on.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd, numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"].str.lower() == "down").astype(int)

feat = pd.DataFrame(index=df.index)
feat["log_impressions"] = np.log1p(df["impressions_90d"])
feat["ctr"] = df["ctr"].fillna(0)
feat["avg_position"] = df["avg_position"].replace(0, np.nan)
feat["avg_position"] = feat["avg_position"].fillna(feat["avg_position"].median())
feat["engagement_rate"] = df["engagement_rate"].fillna(0)
feat["word_count"] = df["word_count"].fillna(df["word_count"].median())
feat["content_age_days"] = df["content_age_days"]
feat["days_since_last_update"] = df["days_since_last_update"]

X = StandardScaler().fit_transform(feat)
k = 6
km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X)
df["cluster"] = km.labels_

archetype_names = {
    5: "Stale Heavyweights", 2: "Aging Page-One (Engagement Gap)", 0: "Young & Slipping",
    1: "Established Performers", 3: "Low-Demand Long-Tail", 4: "Near-Zero-Traffic",
}
df["archetype"] = df["cluster"].map(archetype_names)

playbook_map = {
    "Stale Heavyweights": ("refresh",
        "Long, old-updated, still-visible pages -- refresh content and update dates first; highest decline rate (0.65)."),
    "Aging Page-One (Engagement Gap)": ("refresh_title_and_meta",
        "Ranking near page 1 but stale and thin-engagement -- refresh titles/meta first; decline rate 0.62."),
    "Young & Slipping": ("monitor_closely",
        "Younger pages already at risk despite low staleness -- watch closely, light-touch only; decline rate 0.61."),
    "Established Performers": ("monitor",
        "Mid-position, moderate age, comparatively stable -- routine monitoring only; decline rate 0.41."),
    "Low-Demand Long-Tail": ("low_priority",
        "Deep-position, low-impression pages -- low review priority unless demand shifts; decline rate 0.37."),
    "Near-Zero-Traffic": ("deprioritize",
        "Almost no impressions -- not worth refresh review; lowest decline rate (0.14)."),
}
df["suggested_action"] = df["archetype"].map(lambda a: playbook_map[a][0])
df["action_reason"] = df["archetype"].map(lambda a: playbook_map[a][1])

decline_rate_by_cluster = df.groupby("cluster")["is_declining_label"].mean()
priority_order = decline_rate_by_cluster.sort_values(ascending=False).index.tolist()
df["priority_tier"] = df["cluster"].map({c: i for i, c in enumerate(priority_order)})
queue = df.sort_values(["priority_tier", "impressions_90d"], ascending=[True, False]).reset_index(drop=True)
queue["playbook_rank"] = np.arange(1, len(queue) + 1)

print("Queue built:", len(queue), "rows")
print(queue["suggested_action"].value_counts())
queue[["playbook_rank", "content_id", "archetype", "suggested_action", "action_reason",
       "impressions_90d", "is_declining_label"]].head(10)

Queue built: 30000 rows
suggested_action
monitor_closely           11058
monitor                    7352
refresh_title_and_meta     5925
refresh                    2887
low_priority               2638
deprioritize                140
Name: count, dtype: int64


,playbook_rank,content_id,archetype,suggested_action,action_reason,impressions_90d,is_declining_label
0,1,content_2cb567c3c89b,Stale Heavyweights,refresh,"Long, old-updated, still-visible pages -- refr...",497727,0
1,2,content_2dba2b1f9536,Stale Heavyweights,refresh,"Long, old-updated, still-visible pages -- refr...",443434,0
2,3,content_b28d1efd668f,Stale Heavyweights,refresh,"Long, old-updated, still-visible pages -- refr...",286608,0
3,4,content_813e88069237,Stale Heavyweights,refresh,"Long, old-updated, still-visible pages -- refr...",233561,1
4,5,content_ff94c9b6b411,Stale Heavyweights,refresh,"Long, old-updated, still-visible pages -- refr...",228566,1
5,6,content_66b4046cc144,Stale Heavyweights,refresh,"Long, old-updated, still-visible pages -- refr...",217415,1
6,7,content_b511d4bc4ad2,Stale Heavyweights,refresh,"Long, old-updated, still-visible pages -- refr...",205915,0
7,8,content_c5063073d048,Stale Heavyweights,refresh,"Long, old-updated, still-visible pages -- refr...",192205,0
8,9,content_f02b48f88241,Stale Heavyweights,refresh,"Long, old-updated, still-visible pages -- refr...",181514,0
9,10,content_05e9b4cd9ccf,Stale Heavyweights,refresh,"Long, old-updated, still-visible pages -- refr...",179002,1


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Who uses this:** a FlyRank content strategist deciding which of many pages to review
first in a given week, per client.

**What it's for:** triage ordering, not an automated publishing or de-indexing decision. It
tells a reviewer "look at this group before that group" -- it does not tell them what to write.

**Where it stops being valid:**
- **In-sample numbers overstate it.** ML-09 showed the same archetype approach evaluated
  in-sample (P@20=0.50) drops to P@20=0.35 under an honest client-grouped split -- below even
  the held-out base rate (0.502) at P@50. This playbook's ordering should be treated as
  *decision-support, not a scored guarantee*, and re-validated per new client batch, not
  assumed to transfer automatically.
- **Cluster identity is only moderately stable.** Adjusted Rand Index of 0.59 between two
  reseeded fits (ML-08) means roughly a third of pages could land in a different archetype on
  a re-run -- a page sitting near a cluster boundary should not be treated as permanently
  assigned to its label.
- **It cannot see change.** The features are a single 90-day snapshot; nothing here tracks
  trajectory, so a page that just started declining looks the same as one that has been quiet
  for a year until the next scored batch.
- **It never beat the plain ML-07 rule baseline** on precision@20 or precision@50 in this
  slice (0.50 vs 0.65, 0.46 vs 0.50) -- a reviewer with limited time may get more real hits
  from the simpler rule queue than from this archetype queue alone; the two are complementary
  filters, not a strict upgrade.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Honest (client-grouped) vs in-sample -- from ML-09:")
print("  in-sample:  base_rate=0.542  P@20=0.50  P@50=0.46")
print("  honest:     base_rate=0.502  P@20=0.35  P@50=0.32")
print("Reseed stability (ARI, seed 42 vs 7, from ML-08): 0.59")

Honest (client-grouped) vs in-sample -- from ML-09:
  in-sample:  base_rate=0.542  P@20=0.50  P@50=0.46
  honest:     base_rate=0.502  P@20=0.35  P@50=0.32
Reseed stability (ARI, seed 42 vs 7, from ML-08): 0.59


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**A person must check, before acting on any row:**
- Read the page itself -- does the archetype's story (e.g. "stale but visible") actually match
  what's on the page, or is this one of the wrong picks the error analysis in ML-08/ML-09
  flagged (large, high-impression pages that are not really declining)?
- Confirm `is_declining_label`/trend context lines up with what the reviewer would expect from
  the client's own reporting, since this queue never saw ground truth during clustering.
- Check the archetype's `n` -- `Near-Zero-Traffic` (n=140) is thin; don't generalize its 0.14
  decline rate to a single page with unusually low confidence.

**No-go list -- never automate:**
- Auto-publishing, auto-rewriting, or auto-deleting any page from this queue alone.
- Treating `suggested_action` as a claim about *why* a page is declining, or as a causal fix
  ("refreshing will increase traffic") -- this is a cross-sectional, decision-support ranking,
  not an experiment (see `writing-honest-claims`).
- Sharing row-level exports outside the FlyRank team -- IDs are pseudonymous, not anonymous
  (`DATA_USE.md`), and public outputs stay at the aggregate/chart level.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
no_go_examples = [
    "auto-publish or auto-delete a page from this queue alone",
    "treat suggested_action as a causal explanation for decline",
    "share row-level exports (even pseudonymized) outside the team",
]
for item in no_go_examples:
    print("NO-GO:", item)

NO-GO: auto-publish or auto-delete a page from this queue alone
NO-GO: treat suggested_action as a causal explanation for decline
NO-GO: share row-level exports (even pseudonymized) outside the team


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

**Re-check the archetypes when any of these happen:**
1. **A new data release lands** -- a bigger warehouse pull (this starter CSV is 30,000 rows
   from 32 clients; the full warehouse is larger) should trigger a full re-cluster, not just
   scoring new rows against old centers, since ML-09 showed centers don't transfer cleanly
   across client subsets (P@20 dropped from 0.50 to 0.35 client-out-of-sample).
2. **Precision@20/@50 on a fresh manual audit drops meaningfully below this run's numbers**
   (0.50 / 0.46 in-sample, 0.35 / 0.32 honest) -- if a spot-check of the top of a new queue
   scores much lower than these, the archetypes have likely drifted and need a reseed + relabel.
3. **Reseed ARI drops well below ~0.59** on a scheduled stability check -- would mean the
   groupings are becoming less reproducible than they already are, not more.
4. **The label's own definition changes** (e.g. FlyRank redefines what counts as "declining")
   -- the whole priority ordering is built from `is_declining_label`'s current definition and
   would need to be redone, not just re-scored.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
monitoring_triggers = {
    "new_data_release": "re-cluster from scratch, do not just score new rows against old centers",
    "manual_audit_precision_drop": "below in-sample P@20=0.50 / P@50=0.46 as an early warning; below honest P@20=0.35 / P@50=0.32 as a hard trigger",
    "reseed_ari_drop": "meaningfully below 0.59",
    "label_definition_change": "full re-cluster and re-priority, not incremental",
}
for k_, v in monitoring_triggers.items():
    print(f"{k_}: {v}")

new_data_release: re-cluster from scratch, do not just score new rows against old centers
manual_audit_precision_drop: below in-sample P@20=0.50 / P@50=0.46 as an early warning; below honest P@20=0.35 / P@50=0.32 as a hard trigger
reseed_ari_drop: meaningfully below 0.59
label_definition_change: full re-cluster and re-priority, not incremental


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import json, os

os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/outputs/charts", exist_ok=True)

out_cols = ["playbook_rank", "content_id", "client_id", "archetype", "suggested_action",
            "action_reason", "impressions_90d", "avg_position", "ctr", "word_count",
            "days_since_last_update", "is_declining_label"]
queue[out_cols].to_csv("work/outputs/archetype_action_playbook.csv", index=False)

base_rate = df["is_declining_label"].mean()
p20 = queue["is_declining_label"].head(20).mean()
p50 = queue["is_declining_label"].head(50).mean()

profile = df.groupby("archetype").agg(
    n=("content_id", "count"),
    impressions_median=("impressions_90d", "median"),
    decline_rate=("is_declining_label", "mean"),
).round(3).sort_values("decline_rate", ascending=False)

summary = {
    "n_rows": int(len(df)),
    "base_rate": round(float(base_rate), 3),
    "precision_at_20": round(float(p20), 3),
    "precision_at_50": round(float(p50), 3),
    "archetype_profile": profile.reset_index().to_dict(orient="records"),
    "action_mix": df["suggested_action"].value_counts().to_dict(),
}
with open("work/outputs/playbook_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("Wrote work/outputs/archetype_action_playbook.csv --", len(queue), "rows")
print("Wrote work/outputs/playbook_summary.json")
print(json.dumps(summary, indent=2))

Wrote work/outputs/archetype_action_playbook.csv -- 30000 rows
Wrote work/outputs/playbook_summary.json
{
  "n_rows": 30000,
  "base_rate": 0.542,
  "precision_at_20": 0.5,
  "precision_at_50": 0.46,
  "archetype_profile": [
    {
      "archetype": "Stale Heavyweights",
      "n": 2887,
      "impressions_median": 4599.0,
      "decline_rate": 0.647
    },
    {
      "archetype": "Aging Page-One (Engagement Gap)",
      "n": 5925,
      "impressions_median": 1159.0,
      "decline_rate": 0.617
    },
    {
      "archetype": "Young & Slipping",
      "n": 11058,
      "impressions_median": 555.5,
      "decline_rate": 0.608
    },
    {
      "archetype": "Established Performers",
      "n": 7352,
      "impressions_median": 565.5,
      "decline_rate": 0.412
    },
    {
      "archetype": "Low-Demand Long-Tail",
      "n": 2638,
      "impressions_median": 256.0,
      "decline_rate": 0.369
    },
    {
      "archetype": "Near-Zero-Traffic",
      "n": 140,
      "impressions_medi

## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.